In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


def _find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pricepoint").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root (a directory containing 'pricepoint/') "
        "above the notebook's current working directory."
    )


PROJECT_ROOT = _find_project_root()
PROCESSED_DATA_PATH = str(PROJECT_ROOT / "data" / "02_processed" / "canonical_products_e5.parquet")
df = pd.read_parquet(PROCESSED_DATA_PATH)

df = df.sort_values(by=["supermarket", "canonical_name", "date"]).reset_index(drop=True)

print("Data Loaded and sorted successfully")
df.head()

Data Loaded and sorted successfully


,supermarket,prices,prices_unit,unit,product_name,date,category,own_brand,normalised_name,canonical_name
0,ASDA,2.50,5.4,kg,Light & Free Cherry Greek Style 0% Added Sugar...,2024-01-09,fresh_food,False,light free cherry greek style 0 added sugar fa...,0 fat greek style yogurt
1,ASDA,1.50,3.3,kg,Light & Free Strawberry Greek Style 0% Added S...,2024-01-09,fresh_food,False,light free strawberry greek style 0 added suga...,0 fat greek style yogurt
2,ASDA,1.50,3.3,kg,Light & Free Rasberry Greek Style 0% Added Sug...,2024-01-09,fresh_food,False,light free rasberry greek style 0 added sugar ...,0 fat greek style yogurt
3,ASDA,1.25,2.8,kg,ASDA Greek Style Yogurt with Honey,2024-01-09,fresh_food,True,greek style yogurt with honey,0 fat greek style yogurt
4,ASDA,1.25,2.8,kg,Onken 0% Fat Strawberry,2024-01-09,fresh_food,False,onken 0 fat strawberry,0 fat greek style yogurt


## A. Rooling Price Statistics

In [2]:
windows = [7, 14, 30]

grouped = df.groupby(["supermarket", "canonical_name"])

for window in windows:
    # Rolling mean (price trend)
    df[f'price_rol_mean_{window}d'] = grouped['prices'].transform(
        lambda x: x.rolling(window, min_periods=1).mean()
    )
    # Rolling standard deviation (price volatility)
    df[f'price_rol_std_{window}d'] = grouped['prices'].transform(
        lambda x: x.rolling(window, min_periods=1).std()
    )
    # Rolling min/max (recent price range)
    df[f'price_rol_min_{window}d'] = grouped['prices'].transform(
        lambda x: x.rolling(window, min_periods=1).min()
    )
    df[f'price_rol_max_{window}d'] = grouped['prices'].transform(
        lambda x: x.rolling(window, min_periods=1).max()
    )

# Fill initial NaN values from rolling std with 0
df.fillna({col: 0 for col in df.columns if 'rol_std' in col}, inplace=True)

print("Created rolling price statistics features")

Created rolling price statistics features


# B. Price Momentum and Lag Features

In [3]:
lags = [1, 7]
for lag in lags:
    df[f"price_lag_{lag}d"] = grouped["prices"].transform(lambda x: x.shift(lag))

# Price difference
df["price_diff_1d"] = grouped["prices"].transform(lambda x: x.diff(1))

# Fill NaNs created by shift/diff
df.fillna({col: 0 for col in df.columns if "lag" in col or "diff" in col}, inplace=True)

print("Created price momentum and lag features")

Created price momentum and lag features


# 3. Feature Creation: Capturing Competitive Landscape

## A. Daily Market Price Comparison

In [4]:
daily_market_stats = df.groupby(['canonical_name', 'date'])["prices"].agg(['mean', 'std', 'min', 'max']
).rename(columns={
    'mean': 'market_avg_price',
    'std': 'market_std_price',
    'min': 'market_min_price',
    'max': 'market_max_price'

}).reset_index()

# Merge these market side stats back into main dataframe
df = pd.merge(df, daily_market_stats, on=["canonical_name", "date"], how="left")

# --- Create Competitiveness Scores ---
# Price vs. Market Average: A -ve value means cheaper than average
df["price_vs_market_avg"] = df["prices"] - df["market_avg_price"]

df["price_rank"] = df.groupby(['canonical_name', 'date'])["prices"].rank(method="min")

# binary feature
df["is_cheapest"] = (df["prices"] == df["market_min_price"]).astype(int)

print("Created market competitiveness features.")

Created market competitiveness features.


# 4. Feature Creation: Temporal Features

In [5]:
df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_month"] = df["date"].dt.day
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["month"] = df["date"].dt.month

print("Created temporal features")

Created temporal features


# 5. Save final present

In [6]:
print("Features inspection for one product:")
print(df[df["canonical_name"] == "heinz tomato ketchup bottle"].tail().T)

# Save
FINAL_DATA_PATH = str(PROJECT_ROOT / "data" / "02_processed" / "feature_engineered_data.parquet")
df.to_parquet(FINAL_DATA_PATH)

print(f"\nFeature engineering complete. Data saved to {FINAL_DATA_PATH}")

Features inspection for one product:
Empty DataFrame
Columns: []
Index: [supermarket, prices, prices_unit, unit, product_name, date, category, own_brand, normalised_name, canonical_name, price_rol_mean_7d, price_rol_std_7d, price_rol_min_7d, price_rol_max_7d, price_rol_mean_14d, price_rol_std_14d, price_rol_min_14d, price_rol_max_14d, price_rol_mean_30d, price_rol_std_30d, price_rol_min_30d, price_rol_max_30d, price_lag_1d, price_lag_7d, price_diff_1d, market_avg_price, market_std_price, market_min_price, market_max_price, price_vs_market_avg, price_rank, is_cheapest, day_of_week, day_of_month, week_of_year, month]



Feature engineering complete. Data saved to C:\Project\PricePoint-Dynamics-Decoding-the-UK-Supermarket-Competitive-Landscape-with-Machine-Learning\data\02_processed\feature_engineered_data.parquet
